# Inspect image and mask

Check a 3D image and its binary annotation before calculating any segmentation score.
The default example is **synthetic**: no scans, model weights, or network access are needed.

Run the cells in order. Later, set both file paths below to approved local NIfTI files.
Labels other than 0/1 require an explicit, verified class mapping before using this notebook;
do not convert every nonzero class into pancreas. Nothing here trains or runs a model.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
from matplotlib.colors import ListedColormap

from ml_biomed_project.metrics import dice_score

# Leave both paths as None for the synthetic example. No files are downloaded.
image_path = None
mask_path = None

if (image_path is None) != (mask_path is None):
    raise ValueError("Set both image_path and mask_path, or leave both as None.")

synthetic = image_path is None
if synthetic:
    x, y, z = np.ogrid[:96, :96, :48]
    body = ((x - 48) / 38) ** 2 + ((y - 48) / 32) ** 2 + ((z - 24) / 20) ** 2
    organ = ((x - 49) / 20) ** 2 + ((y - 45) / 9) ** 2 + ((z - 24) / 7) ** 2 <= 1
    rng = np.random.default_rng(0)
    values = (70 * np.exp(-body) + 45 * organ + rng.normal(0, 2, organ.shape)).astype(np.float32)
    affine = np.diag([1.5, 1.5, 3.0, 1.0])
    image = nib.Nifti1Image(values, affine)
    mask = nib.Nifti1Image(organ.astype(np.uint8), affine)
    image.header.set_xyzt_units("mm")
    mask.header.set_xyzt_units("mm")
    print("Synthetic example: not a medical scan or a model prediction.")
else:
    image = nib.load(Path(image_path).expanduser())
    mask = nib.load(Path(mask_path).expanduser())

## Check geometry and values

Matching array sizes alone do not establish alignment: the affine matrices must agree too.
These checks reject mismatches; they do not resample, reorient, or modify labels. Agreement
of file geometry also does not prove that the annotation belongs to the correct patient.
An unknown spatial unit remains unknown and must be checked before real-data analysis.

In [ ]:
def check_image_and_mask(image, mask):
    """Validate this notebook's 3D binary-mask inputs without changing their geometry."""
    if len(image.shape) != 3 or len(mask.shape) != 3:
        raise ValueError("Expected a 3D image and a 3D mask.")
    if image.shape != mask.shape:
        raise ValueError("Image and mask shapes do not match.")
    for name, volume in (("Image", image), ("Mask", mask)):
        spacing = np.asarray(volume.header.get_zooms()[:3])
        if not np.all(np.isfinite(spacing)) or np.any(spacing <= 0):
            raise ValueError(f"{name} voxel spacing must be finite and positive.")
        if not np.all(np.isfinite(volume.affine)) or np.isclose(
            np.linalg.det(volume.affine[:3, :3]), 0
        ):
            raise ValueError(f"{name} affine must be finite and nonsingular.")
    if not np.allclose(image.affine, mask.affine, rtol=0, atol=1e-5):
        raise ValueError("Image and mask affines do not match; do not calculate Dice yet.")
    if image.header.get_xyzt_units()[0] != mask.header.get_xyzt_units()[0]:
        raise ValueError("Image and mask spatial units do not match.")
    image_values = image.get_fdata(dtype=np.float32)
    mask_values = np.asarray(mask.dataobj)
    if not np.all(np.isfinite(image_values)):
        raise ValueError("Image contains nonfinite values.")
    if not np.all((mask_values == 0) | (mask_values == 1)):
        raise ValueError("Mask must contain only binary 0/1 values; verify the label mapping.")
    return image_values, mask_values


image_values, mask_values = check_image_and_mask(image, mask)
print("Shape:", image.shape)
print("Voxel spacing:", tuple(float(value) for value in image.header.get_zooms()[:3]))
print("Spatial unit:", image.header.get_xyzt_units()[0])
print("Increasing-axis directions:", nib.aff2axcodes(image.affine))
print("Affine:\n", image.affine)
print("Mask values:", np.unique(mask_values))
print("Foreground voxels:", np.count_nonzero(mask_values))
if not np.any(mask_values):
    print("Empty annotation: review this case before using it for a pancreas pilot.")

## Inspect the overlay

The displayed slice along array axis 2 contains the most annotation voxels (or the middle slice for
an empty mask). That label-informed choice is **only for visual inspection**: Dice still
uses the complete volume. The display preserves in-plane voxel proportions and shows native
array axes, not a standardized radiological view. No inference crop is derived from labels.

In [ ]:
slice_counts = np.count_nonzero(mask_values, axis=(0, 1))
slice_index = int(np.argmax(slice_counts)) if np.any(slice_counts) else image.shape[2] // 2
slice_image = image_values[:, :, slice_index].T
slice_mask = mask_values[:, :, slice_index].T
spacing = image.header.get_zooms()[:3]

fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
limits = np.percentile(image_values, [1, 99])
for ax, title in zip(axes, ("Image", "Image + reference annotation"), strict=True):
    ax.imshow(
        slice_image,
        cmap="gray",
        origin="lower",
        aspect=float(spacing[1] / spacing[0]),
        vmin=limits[0],
        vmax=limits[1],
        interpolation="nearest",
    )
    ax.set(title=title, xlabel="Array axis 0 (voxels)", ylabel="Array axis 1 (voxels)")
axes[1].imshow(
    np.ma.masked_where(slice_mask == 0, slice_mask),
    cmap=ListedColormap(["#ff9500"]),
    origin="lower",
    aspect=float(spacing[1] / spacing[0]),
    alpha=0.45,
    interpolation="nearest",
)
fig.suptitle(f"{'Synthetic example' if synthetic else 'Local image'} — slice {slice_index}")
plt.show()

## Demonstrate Dice and failed alignment checks

For synthetic data only, shift the artificial mask by three voxels to demonstrate the metric.
This is not a segmentation method or an experiment result. Two empty masks return `NaN`
(undefined), and should be reported explicitly rather than silently omitted from a study.
The function checks arrays; the geometry checks above must precede evaluation of real files.

In [ ]:
if synthetic:
    shifted_mask = np.roll(mask_values, 3, axis=0)
    print(f"Synthetic shifted-mask whole-volume Dice: {dice_score(shifted_mask, mask_values):.3f}")
    print("Two empty masks:", dice_score(np.zeros_like(mask_values), np.zeros_like(mask_values)))

    shifted_affine = mask.affine.copy()
    shifted_affine[0, 3] += 3.0
    bad_affine_mask = nib.Nifti1Image(mask_values, shifted_affine)
    bad_shape_mask = nib.Nifti1Image(mask_values[:-1], mask.affine)
    for defect, bad_mask in (("shifted affine", bad_affine_mask), ("wrong shape", bad_shape_mask)):
        try:
            check_image_and_mask(image, bad_mask)
        except ValueError as error:
            print(f"Correctly rejected {defect}: {error}")
        else:
            raise AssertionError(f"The notebook failed to detect {defect}.")
else:
    print("No model prediction supplied; no real-data Dice is calculated.")